In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

df = pd.read_csv("results/benchmark_memory_results_isambard.csv")
df['speedup'] = df['exact_time_sec'] / df['mcmc_avg_time_sec']

# ============================================================
# STATS
# ============================================================
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total configurations: {len(df)}")
print(f"Hidden variables range: {df['n_hidden'].min()} – {df['n_hidden'].max()}")
print(f"Partition sizes range:  {df['max_partition_size'].min()} – {df['max_partition_size'].max()}")
print(f"Exact SDP success rate: {df['exact_success'].sum()}/{len(df)}")
print(f"MCMC success rate:      {df['mcmc_success'].sum()}/{len(df)}")

print("\n" + "=" * 60)
print("TIME ANALYSIS")
print("=" * 60)
print(f"\nExact SDP time (sec):")
print(f"  min:    {df['exact_time_sec'].min():.4f}")
print(f"  max:    {df['exact_time_sec'].max():.4f}")
print(f"  median: {df['exact_time_sec'].median():.4f}")
print(f"  mean:   {df['exact_time_sec'].mean():.4f}")
print(f"\nMCMC avg time (sec):")
print(f"  min:    {df['mcmc_avg_time_sec'].min():.4f}")
print(f"  max:    {df['mcmc_avg_time_sec'].max():.4f}")
print(f"  median: {df['mcmc_avg_time_sec'].median():.4f}")
print(f"  mean:   {df['mcmc_avg_time_sec'].mean():.4f}")

crossover = df[df['speedup'] >= 1.0].iloc[0]
print(f"\nCrossover point (exact becomes slower than MCMC):")
print(f"  n_hidden={crossover['n_hidden']} | partition_size={crossover['max_partition_size']} | speedup={crossover['speedup']:.2f}x")
print(f"\nMax speedup at partition {df['max_partition_size'].max()}: {df['speedup'].max():.1f}x")

print("\n" + "=" * 60)
print("MEMORY ANALYSIS")
print("=" * 60)
print(f"\nExact SDP peak memory (MB):")
print(f"  min:    {df['exact_peak_memory_mb'].min():.2f}")
print(f"  max:    {df['exact_peak_memory_mb'].max():.2f}")
print(f"  median: {df['exact_peak_memory_mb'].median():.2f}")
print(f"\nMCMC peak memory (MB):")
print(f"  min:    {df['mcmc_peak_memory_mb'].min():.2f}")
print(f"  max:    {df['mcmc_peak_memory_mb'].max():.2f}")
print(f"  median: {df['mcmc_peak_memory_mb'].median():.2f}")

mem_crossover = df[df['exact_peak_memory_mb'] >= df['mcmc_peak_memory_mb']].iloc[0]
print(f"\nMemory crossover (exact uses more RAM than MCMC):")
print(f"  n_hidden={mem_crossover['n_hidden']} | partition_size={mem_crossover['max_partition_size']}")

print("\n" + "=" * 60)
print("EXPONENTIAL GROWTH ANALYSIS (EXACT SDP)")
print("=" * 60)
safe = df[df['exact_success'] == True].copy()
log_mem = np.log2(safe['exact_peak_memory_mb'])
slope_mem, intercept_mem, r_mem, _, _ = stats.linregress(safe['max_partition_size'], log_mem)
print(f"\nMemory vs partition size (log2):")
print(f"  doubling rate: every {1/slope_mem:.2f} partition units (expected: 1.0)")
print(f"  R²: {r_mem**2:.4f}")

log_time = np.log2(safe['exact_time_sec'])
slope_time, intercept_time, r_time, _, _ = stats.linregress(safe['max_partition_size'], log_time)
print(f"\nTime vs partition size (log2):")
print(f"  doubling rate: every {1/slope_time:.2f} partition units")
print(f"  R²: {r_time**2:.4f}")

print("\n" + "=" * 60)
print("MCMC ACCURACY ANALYSIS")
print("=" * 60)
accurate = df[df['mcmc_avg_estimate'] == 1.0]
inaccurate = df[df['mcmc_avg_estimate'] != 1.0]
print(f"Perfect MCMC estimate (1.0): {len(accurate)}/{len(df)}")
print(f"Inaccurate MCMC estimate:    {len(inaccurate)}/{len(df)}")
for _, row in inaccurate.iterrows():
    err = abs(1.0 - row['mcmc_avg_estimate'])
    print(f"  n_hidden={int(row['n_hidden'])} | partition={int(row['max_partition_size'])} | "
          f"estimate={row['mcmc_avg_estimate']:.4f} | abs_error={err:.4f}")

print("\n" + "=" * 60)
print("PRACTICAL THRESHOLDS")
print("=" * 60)
for mem_limit in [64, 256, 1024, 4096]:
    safe_rows = df[df['exact_peak_memory_mb'] <= mem_limit]
    if len(safe_rows) > 0:
        max_safe = safe_rows.iloc[-1]
        print(f"  Memory limit {mem_limit:5d} MB → max partition={int(max_safe['max_partition_size'])} | max n_hidden={int(max_safe['n_hidden'])}")
for time_limit in [1, 5, 30, 60]:
    safe_rows = df[df['exact_time_sec'] <= time_limit]
    if len(safe_rows) > 0:
        max_safe = safe_rows.iloc[-1]
        print(f"  Time limit   {time_limit:5d} sec → max partition={int(max_safe['max_partition_size'])} | max n_hidden={int(max_safe['n_hidden'])}")

# ============================================================
# PLOTS
# ============================================================
BLUE  = '#185FA5'
AMBER = '#B85C00'
GREEN = '#3B6D11'
PURP  = '#534AB7'
GRAY  = '#888780'

fig = plt.figure(figsize=(14, 18))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

x = df['n_hidden']

# ── 1. Time comparison (log) ──────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.semilogy(x, df['exact_time_sec'], color=BLUE, lw=2, marker='o', ms=3, label='Exact SDP')
ax1.semilogy(x, df['mcmc_avg_time_sec'], color=AMBER, lw=2, ls='--', marker='s', ms=3, label='MCMC')
crossover_idx = df[df['speedup'] >= 1.0].index[0]
ax1.axvline(df.loc[crossover_idx, 'n_hidden'], color=GRAY, ls=':', lw=1.2, label=f"crossover n={int(df.loc[crossover_idx,'n_hidden'])}")
ax1.set_xlabel('Hidden variables', fontsize=11)
ax1.set_ylabel('Time (sec, log scale)', fontsize=11)
ax1.set_title('Execution time: exact SDP vs MCMC', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── 2. Memory comparison (log) ───────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.semilogy(x, df['exact_peak_memory_mb'], color=BLUE, lw=2, marker='o', ms=3, label='Exact SDP')
ax2.semilogy(x, df['mcmc_peak_memory_mb'], color=AMBER, lw=2, ls='--', marker='s', ms=3, label='MCMC')
mem_cross_idx = df[df['exact_peak_memory_mb'] >= df['mcmc_peak_memory_mb']].index[0]
ax2.axvline(df.loc[mem_cross_idx, 'n_hidden'], color=GRAY, ls=':', lw=1.2, label=f"crossover n={int(df.loc[mem_cross_idx,'n_hidden'])}")
ax2.set_xlabel('Hidden variables', fontsize=11)
ax2.set_ylabel('Peak memory (MB, log scale)', fontsize=11)
ax2.set_title('Peak memory: exact SDP vs MCMC', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ── 3. Speedup factor ────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
colors_bar = [PURP if s > 10 else '#7F77DD' if s > 1 else '#AFA9EC' for s in df['speedup']]
ax3.bar(x, df['speedup'], color=colors_bar, width=0.7)
ax3.axhline(1.0, color=AMBER, ls='--', lw=1.2, label='break-even (1×)')
ax3.set_xlabel('Hidden variables', fontsize=11)
ax3.set_ylabel('Speedup factor (×)', fontsize=11)
ax3.set_title('MCMC speedup over exact SDP', fontsize=12)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

# ── 4. Partition size vs n_hidden ────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.bar(x, df['max_partition_size'], color=df['max_partition_size'].apply(
    lambda p: '#A32D2D' if p >= 24 else '#B85C00' if p >= 20 else BLUE if p >= 15 else GRAY), width=0.7)
ax4.set_xlabel('Hidden variables', fontsize=11)
ax4.set_ylabel('Largest partition size', fontsize=11)
ax4.set_title('Largest partition size vs hidden variables', fontsize=12)
ax4.grid(True, alpha=0.3, axis='y')
for label, color in [('≥24 (critical)', '#A32D2D'), ('≥20 (risky)', '#B85C00'),
                      ('≥15 (moderate)', BLUE), ('<15 (safe)', GRAY)]:
    ax4.bar(0, 0, color=color, label=label)
ax4.legend(fontsize=8)



fig.suptitle('Exact SDP vs MCMC benchmark analysis', fontsize=14, fontweight='bold', y=0.98)
plt.savefig('benchmark_analysis.png', dpi=150, bbox_inches='tight', facecolor='white')
print("\nPlot saved to benchmark_analysis.png")

DATASET OVERVIEW
Total configurations: 31
Hidden variables range: 1 – 31
Partition sizes range:  1 – 24
Exact SDP success rate: 31/31
MCMC success rate:      31/31

TIME ANALYSIS

Exact SDP time (sec):
  min:    0.0251
  max:    126.4396
  median: 0.0263
  mean:   4.4068

MCMC avg time (sec):
  min:    1.6405
  max:    1.8819
  median: 1.7676
  mean:   1.7516

Crossover point (exact becomes slower than MCMC):
  n_hidden=29 | partition_size=22 | speedup=1.38x

Max speedup at partition 24: 71.5x

MEMORY ANALYSIS

Exact SDP peak memory (MB):
  min:    0.18
  max:    2304.08
  median: 0.21

MCMC peak memory (MB):
  min:    1.32
  max:    4.49
  median: 2.34

Memory crossover (exact uses more RAM than MCMC):
  n_hidden=22 | partition_size=15

EXPONENTIAL GROWTH ANALYSIS (EXACT SDP)

Memory vs partition size (log2):
  doubling rate: every 1.82 partition units (expected: 1.0)
  R²: 0.8588

Time vs partition size (log2):
  doubling rate: every 3.22 partition units
  R²: 0.6294

MCMC ACCURACY A